In [7]:
import pandas as pd

file_loc = '/mnt/disks/filedisk2a/Qingcheng/'
train_df   = pd.read_csv(file_loc + 'train_df.csv')
holdout_df = pd.read_csv(file_loc + 'holdout_df.csv')
# exposure = pd.read_csv(file_loc + 'exposure_dthr.csv')
# # # stream bid_dthr.csv in chunks, keep only short_profit > 0 (avoids loading the whole file)
# bids = pd.concat(
#     (chunk[chunk['short_profit'] < 0]
#      for chunk in pd.read_csv(file_loc + 'bid_dthr.csv', chunksize=500_000)),
#     ignore_index=True,
# )
# print('train_df  :', train_df.shape)
# print('holdout_df:', holdout_df.shape)
# train_df.head()

In [5]:
bids.drop_duplicates(['dt','incdec','node_num']).groupby(['dt'])['profit_congestion'].sum().reset_index().sort_values(['profit_congestion'])[:20]

,dt,profit_congestion
1492,2025-08-06,-405319.34808
1187,2024-09-27,-336011.84652
799,2023-08-28,-238273.83238
1385,2025-04-21,-152991.62315
1458,2025-07-03,-140559.69367
1521,2025-09-04,-137245.16800
943,2024-01-24,-123385.79668
1225,2024-11-04,-115466.29978
1476,2025-07-21,-113406.96762
1177,2024-09-17,-105191.89826


In [20]:
# top-3 worst short_profit rows per worst-congestion day, across train_df + holdout_df
# 1) the 20 worst congestion-loss days
worst_dts = (bids.drop_duplicates(['dt', 'incdec', 'node_num'])
                 .groupby('dt')['profit_congestion'].sum()
                 .sort_values().head(20)
                 .index.astype(str).tolist())

# 2) pull those dates from both splits (a date lives in exactly one of them)
both = pd.concat([train_df.assign(split='train'),
                  holdout_df.assign(split='holdout')], ignore_index=True)
both['dt'] = both['dt'].astype(str)
sel = both[both['dt'].isin(worst_dts)]

# 3) per date, the 3 rows with the lowest (most negative) short_profit
top3 = (sel.sort_values(['dt', 'short_profit'])
           .groupby('dt', group_keys=False).head(1)
           .sort_values(['dt', 'short_profit']))

show = [c for c in ['split', 'dt', 'constraint_family_num', 'monitored_name',
                    'short_profit', 'long_profit', 'FlowRatio', 'KV_group', 'rtm_sum_7d', 'rtm_sum_1m', 'dam_sum_7d','rt_spike_1m', 'ShadowPrice_sum', 'rtda_std_1m','rtm','dam']
        if c in top3.columns]
print('worst-congestion dates found in splits:', sel['dt'].nunique(), '/', len(worst_dts))
top3[show]

worst-congestion dates found in splits: 20 / 20


,split,dt,constraint_family_num,monitored_name,short_profit,long_profit,FlowRatio,KV_group,rtm_sum_7d,rtm_sum_1m,dam_sum_7d,rt_spike_1m,ShadowPrice_sum,rtda_std_1m,rtm,dam
14431,train,2021-08-15,61392,lnsheffld-sand_spg,-1.410259e+06,1.564070e+05,0.515111,03_138,-2481.4514,-51309.8264,-18776.1281,-2565.491320,0.000000,3300.837947,-2861.5885,-648.1188
634878,holdout,2022-11-01,1064088,lnsp-frnk3-quincy_s,-1.590232e+06,6.816361e+03,1.000000,02_<=115,-7317.6761,-14616.9161,-7574.2843,-974.461073,-9568.065922,1065.764450,-11846.7617,-1823.2914
166492,train,2023-03-15,948203,lnkinze-kinzeg1,-4.172678e+05,2.500411e+05,1.000000,03_138,0.0000,-6407.9373,-200.3203,-587.394252,-1214.828253,745.065615,-3150.8693,-864.8721
222851,train,2023-08-28,146,lnchar_ck-watford,-4.438910e+06,2.875097e+05,1.000000,04_345,-39225.9820,-195764.9758,-66511.4134,-0.000000,-730.204375,7223.853510,-24660.8043,-9619.6841
224486,train,2023-08-31,1082,xfmrturk_pp-turk_pp,-3.921823e+05,1.491490e+06,0.506696,03_138,-4975.8997,-6728.8788,-212.6477,-448.591920,0.000000,668.601880,-5785.1198,-507.6961
677316,holdout,2023-10-05,533258,NaN,-2.493460e+06,2.071987e+05,NaN,02_<=115,0.0000,0.0000,0.0000,0.000000,NaN,0.000000,-8542.2087,-2428.0833
276417,train,2024-01-24,1066,xfmrsub975-sub975,-1.446998e+06,2.196766e+04,0.694529,04_345,-1717.9009,-3567.0490,-4515.7040,-1664.622867,0.000000,613.216412,-4522.6269,-1607.0569
325628,train,2024-05-31,904,NaN,-2.713538e+06,3.920698e+05,NaN,03_138,-0.4888,-23.0755,-336.1784,-6.922650,NaN,83.099172,-6978.4395,-267.9770
334568,train,2024-06-17,233814,lnmidj-lec,-6.043667e+05,0.000000e+00,0.945252,02_<=115,0.0000,0.0000,-1041.3317,0.000000,0.000000,190.120287,-10277.4341,-346.4865
365173,train,2024-09-17,937,xfmrcimarron-cimarron,-2.374278e+06,7.791602e+05,1.000000,04_345,-607.1390,-28384.4867,-2286.4906,-1237.272497,-14066.193323,1845.289941,-17240.6375,-5898.9101


In [21]:
# replace top3's monitored_name using the canonical constraint_family_num -> monitored_name table
import sys
sys.path.append("/var/www/python/Qingcheng/nighthawk/")
from nighthawk.data.network.constraint import Constraint

OPEX = 'SPP'
# oops -> monitored_name (DA+RT), then oops -> family, then family -> representative monitored_name
det = pd.concat([
    Constraint(market=OPEX).get_constraint_details(da_or_rt="DA"),
    Constraint(market=OPEX).get_constraint_details(da_or_rt="RT"),
], ignore_index=True)[["oops_constraint_num", "monitored_clean"]].drop_duplicates()
fam = Constraint(det[["oops_constraint_num"]].drop_duplicates(), market=OPEX).get_constraint_family_num()
fam_name = (fam.merge(det, on="oops_constraint_num", how="left")
              .dropna(subset=["monitored_clean"])
              .groupby("constraint_family_num")["monitored_clean"]
              .agg(lambda s: s.value_counts().index[0])
              .reset_index()
              .rename(columns={"monitored_clean": "monitored_name_new"}))

# merge onto top3 and REPLACE the monitored_name column
fam_name["constraint_family_num"] = fam_name["constraint_family_num"].astype("int64")
top3["constraint_family_num"]     = top3["constraint_family_num"].astype("int64")
top3 = top3.merge(fam_name, on="constraint_family_num", how="left")
top3["monitored_name"] = top3["monitored_name_new"].fillna(top3.get("monitored_name"))
top3 = top3.drop(columns="monitored_name_new")

print("name remapped:", f"{top3['monitored_name'].notna().mean():.0%} of {len(top3)} rows")
top3[show]

name remapped: 100% of 20 rows


,split,dt,constraint_family_num,monitored_name,short_profit,long_profit,FlowRatio,KV_group,rtm_sum_7d,rtm_sum_1m,dam_sum_7d,rt_spike_1m,ShadowPrice_sum,rtda_std_1m,rtm,dam
0,train,2021-08-15,61392,lnsheffld-sand_spg,-1.410259e+06,1.564070e+05,0.515111,03_138,-2481.4514,-51309.8264,-18776.1281,-2565.491320,0.000000,3300.837947,-2861.5885,-648.1188
1,holdout,2022-11-01,1064088,lnsp-frnk3-quincy_s,-1.590232e+06,6.816361e+03,1.000000,02_<=115,-7317.6761,-14616.9161,-7574.2843,-974.461073,-9568.065922,1065.764450,-11846.7617,-1823.2914
2,train,2023-03-15,948203,lnkinze-kinzeg1,-4.172678e+05,2.500411e+05,1.000000,03_138,0.0000,-6407.9373,-200.3203,-587.394252,-1214.828253,745.065615,-3150.8693,-864.8721
3,train,2023-08-28,146,lnchar_ck-watford,-4.438910e+06,2.875097e+05,1.000000,04_345,-39225.9820,-195764.9758,-66511.4134,-0.000000,-730.204375,7223.853510,-24660.8043,-9619.6841
4,train,2023-08-31,1082,xfmrturk_pp-turk_pp,-3.921823e+05,1.491490e+06,0.506696,03_138,-4975.8997,-6728.8788,-212.6477,-448.591920,0.000000,668.601880,-5785.1198,-507.6961
5,holdout,2023-10-05,533258,xfmrgrndfks-grndfks,-2.493460e+06,2.071987e+05,NaN,02_<=115,0.0000,0.0000,0.0000,0.000000,NaN,0.000000,-8542.2087,-2428.0833
6,train,2024-01-24,1066,xfmrsub975-sub975,-1.446998e+06,2.196766e+04,0.694529,04_345,-1717.9009,-3567.0490,-4515.7040,-1664.622867,0.000000,613.216412,-4522.6269,-1607.0569
7,train,2024-05-31,904,multi-elementconstraint.spsnmties.spsnmties,-2.713538e+06,3.920698e+05,NaN,03_138,-0.4888,-23.0755,-336.1784,-6.922650,NaN,83.099172,-6978.4395,-267.9770
8,train,2024-06-17,233814,lnmidj-lec,-6.043667e+05,0.000000e+00,0.945252,02_<=115,0.0000,0.0000,-1041.3317,0.000000,0.000000,190.120287,-10277.4341,-346.4865
9,train,2024-09-17,937,xfmrcimarron-cimarron,-2.374278e+06,7.791602e+05,1.000000,04_345,-607.1390,-28384.4867,-2286.4906,-1237.272497,-14066.193323,1845.289941,-17240.6375,-5898.9101


# 1. multi-element does not have flow ratio 
# 2. kv 03_138 showcases some flowratio is not good and represent a lot 
# 3. rtm_sum_1m and rtda_std_1m high 
# quantile of the variables, know the constraint other reasons. 


In [14]:
# same top-3 rows, showing the requested feature variables
show2 = [c for c in ['split', 'dt', 'constraint_family_num', 'monitored_name', 'short_profit',
                     'rtm_sum_7d', 'rtm_sum_1m', 'ShadowPrice_sum', 'rtda_std_1m']
         if c in top3.columns]
missing = [c for c in ['rtm_sum_7d', 'rtm_sum_1m', 'ShadowPrice_sum', 'rtda_std_1m']
           if c not in top3.columns]
if missing:
    print('missing columns:', missing)
top3[show2]

,split,dt,constraint_family_num,monitored_name,short_profit,rtm_sum_7d,rtm_sum_1m,ShadowPrice_sum,rtda_std_1m
14431,train,2021-08-15,61392,lnsheffld-sand_spg,-1.410259e+06,-2481.4514,-51309.8264,0.000000,3300.837947
634878,holdout,2022-11-01,1064088,lnsp-frnk3-quincy_s,-1.590232e+06,-7317.6761,-14616.9161,-9568.065922,1065.764450
166492,train,2023-03-15,948203,lnkinze-kinzeg1,-4.172678e+05,0.0000,-6407.9373,-1214.828253,745.065615
222851,train,2023-08-28,146,lnchar_ck-watford,-4.438910e+06,-39225.9820,-195764.9758,-730.204375,7223.853510
224486,train,2023-08-31,1082,xfmrturk_pp-turk_pp,-3.921823e+05,-4975.8997,-6728.8788,0.000000,668.601880
677316,holdout,2023-10-05,533258,NaN,-2.493460e+06,0.0000,0.0000,NaN,0.000000
276417,train,2024-01-24,1066,xfmrsub975-sub975,-1.446998e+06,-1717.9009,-3567.0490,0.000000,613.216412
325628,train,2024-05-31,904,NaN,-2.713538e+06,-0.4888,-23.0755,NaN,83.099172
334568,train,2024-06-17,233814,lnmidj-lec,-6.043667e+05,0.0000,0.0000,0.000000,190.120287
365173,train,2024-09-17,937,xfmrcimarron-cimarron,-2.374278e+06,-607.1390,-28384.4867,-14066.193323,1845.289941


In [10]:
train_df.sort_values(['short_profit'])[:20]

,constraint_family_num,dt,long_profit,short_profit,long_profit_sum_3d,short_profit_sum_3d,long_profit_sum_7d,short_profit_sum_7d,long_profit_sum_1m,short_profit_sum_1m,...,rt_nz_cnt_3m,rt_nz_cnt_1y,rt_nz_cnt_3y,rt_spike_7d,rt_spike_1m,rt_spike_3m,rt_spike_1y,rt_spike_3y,KV,KV_group
272179,594682,2024-01-07,1.955651e+05,-7.897300e+06,5.663392e+04,-5.525440e+05,5.663392e+04,-5.525440e+05,3.228892e+03,-5.008465e+05,...,10.0,30.0,74.0,-2713.595229,-3060.332397,-1032.007831,-626.265986,-455.338596,230.0,04_345
480321,858,2025-08-06,4.464750e+05,-7.497350e+06,-1.289327e+04,2.576179e+05,-5.380438e+04,3.854568e+05,-4.859382e+05,1.078640e+03,...,17.0,17.0,17.0,0.000000,-5864.531010,-4719.419831,-5547.465908,-5728.130507,138.0,03_138
18292,34192,2021-08-28,8.665428e+05,-5.236005e+06,6.811200e+04,-7.497731e+04,1.073421e+06,-2.840727e+06,1.013141e+06,-2.776881e+06,...,12.0,12.0,20.0,-0.000000,-1050.519327,-1266.978223,-1413.835552,-1147.083340,138.0,03_138
222851,146,2023-08-28,2.875097e+05,-4.438910e+06,-6.094366e+05,2.234393e+05,-5.177138e+05,1.726122e+05,2.474063e+06,-3.793186e+06,...,67.0,224.0,431.0,-0.000000,-0.000000,-968.618683,-982.169134,-1124.768195,230.0,04_345
391000,4674,2024-11-04,9.730528e+05,-4.349045e+06,7.950128e+04,-3.717576e+05,1.524731e+06,-1.136586e+06,7.479160e+05,-7.619924e+05,...,38.0,114.0,170.0,-0.000000,-325.078529,-718.790116,-749.250186,-842.737323,138.0,03_138
335526,904,2024-06-19,1.486245e+06,-3.619023e+06,9.467655e+04,-6.675120e+04,2.308235e+05,-1.190407e+05,1.374253e+06,-3.647771e+06,...,41.0,69.0,213.0,-29.833255,-425.179647,-932.765030,-1045.988457,-836.075549,NaN,03_138
473073,858,2025-07-21,1.092146e+05,-3.247985e+06,-1.393477e+04,4.465480e+05,-6.270073e+04,2.562363e+06,-6.526941e+05,-6.067754e+05,...,13.0,13.0,13.0,0.000000,-4199.096100,-4728.379244,-5329.836486,-5461.063521,138.0,03_138
73355,572,2022-04-22,9.086355e+05,-3.247233e+06,-5.751830e+05,1.362408e+06,-9.578800e+05,1.830534e+06,-3.717203e+06,3.540119e+06,...,76.0,172.0,492.0,-0.000000,-619.382656,-305.626931,-846.025634,-645.693497,161.0,04_345
274242,54,2024-01-12,1.861753e+06,-3.050114e+06,4.605118e+05,-1.308412e+05,5.606452e+05,-1.803394e+05,2.504394e+06,-1.810997e+06,...,50.0,92.0,158.0,-1291.785707,-845.348598,-1280.639957,-2126.126886,-2304.016645,161.0,04_345
548696,1039619,2026-01-16,1.516780e+05,-2.926611e+06,4.972776e+04,-2.767342e+04,4.972776e+04,-2.767342e+04,4.972776e+04,-2.767342e+04,...,2.0,8.0,17.0,-1366.121071,-1785.064867,-1870.067956,-839.693526,-1096.499412,161.0,04_345


In [29]:
bids.query("dt == '2025-08-06' and constraintFamilyNum == 858")['short_profit'].sum()

np.float64(-7497349.870521045)

In [30]:
bids.groupby(['dt', 'constraintFamilyNum'])['short_profit'].sum().reset_index()['short_profit'].quantile([0.05, 0.01, 0.05, 0.1, 0.7, 0.9, 0.99])

0.05   -250342.538888
0.01   -715586.026896
0.05   -250342.538888
0.10   -133566.282851
0.70     -2619.805184
0.90      -281.034020
0.99        -8.165545
Name: short_profit, dtype: float64

In [9]:
bids.query('dt == "2025-07-15" and node_num == 1189')

,Unnamed: 0,dt,node_num,incdec,bid_mw,clear_mw,profit,profit_congestion,node_name,bid_mw_inc,...,constraintFamilyNum,dfax,long_bid_mw,short_bid_mw,long_clear_mw,short_clear_mw,dam,rtm,long_profit,short_profit
510808,72965830,2025-07-15,1189,Decrement,383.4,212.5,-23605.51512,-27581.054,OKGE.FRONTIER,0.0,...,281,-0.01850,0.0,7.092900,0.0,3.931250,0.0000,-112.9691,0.0,-444.109774
510809,72965850,2025-07-15,1189,Decrement,383.4,212.5,-23605.51512,-27581.054,OKGE.FRONTIER,0.0,...,421,-0.08462,0.0,32.443308,0.0,17.981750,-276.7274,-1801.1301,0.0,-27411.428251
510810,72965888,2025-07-15,1189,Decrement,383.4,212.5,-23605.51512,-27581.054,OKGE.FRONTIER,0.0,...,748,-0.01870,0.0,7.169580,0.0,3.973750,-640.5805,-3854.3942,0.0,-12770.892190
510811,72965899,2025-07-15,1189,Decrement,383.4,212.5,-23605.51512,-27581.054,OKGE.FRONTIER,0.0,...,845,-0.01310,0.0,5.022540,0.0,2.783750,-740.3934,-6704.6811,0.0,-16603.085885
510812,72965915,2025-07-15,1189,Decrement,383.4,212.5,-23605.51512,-27581.054,OKGE.FRONTIER,0.0,...,924,-0.02359,0.0,9.044406,0.0,5.012875,0.0000,-627.7851,0.0,-3147.008233
510813,72965934,2025-07-15,1189,Decrement,383.4,212.5,-23605.51512,-27581.054,OKGE.FRONTIER,0.0,...,1012,-0.02209,0.0,8.469306,0.0,4.694125,-37.4133,-1006.5131,0.0,-4549.075599
510814,72965938,2025-07-15,1189,Decrement,383.4,212.5,-23605.51512,-27581.054,OKGE.FRONTIER,0.0,...,1037,-0.05000,0.0,19.170000,0.0,10.625000,-233.5591,-241.9243,0.0,-88.880250
510815,72965991,2025-07-15,1189,Decrement,383.4,212.5,-23605.51512,-27581.054,OKGE.FRONTIER,0.0,...,236111,-0.87586,0.0,335.804724,0.0,186.120250,-640.8511,-2349.9405,0.0,-318096.146400
510816,72966007,2025-07-15,1189,Decrement,383.4,212.5,-23605.51512,-27581.054,OKGE.FRONTIER,0.0,...,368276,-0.02498,0.0,9.577332,0.0,5.308250,0.0000,-45.0080,0.0,-238.913716


In [2]:
# merge daily physical variables (wind / load / genoutage / ice price forecast) onto the splits
import sys
sys.path.append("/var/www/python/Prod/nighthawk/")
import pandas as pd, numpy as np
from nighthawk.data.pipeline.common_functions import wind, load, genoutage
from nighthawk.data.pipeline.var_handler import ice_elec_price_vh

OPEX = 'SPP'

# date range covering both splits
dts = pd.concat([train_df['dt'], holdout_df['dt']]).astype(str)
start_dt, end_dt = dts.min(), dts.max()
print('phys date range:', start_dt, '->', end_dt)

def _daily(df, col, name):
    """hourly forecast -> daily mean, dt as 'YYYY-MM-DD' string"""
    df = df.copy()
    df['dt'] = pd.to_datetime(df['dt']).dt.strftime('%Y-%m-%d')
    return df.groupby('dt', as_index=False)[col].mean().rename(columns={col: name})

# --- wind / load / genoutage daily forecast ---
wind_df  = wind.Wind(OPEX).get_total_wind(start_dt, end_dt, var_spec=['f'], impute=True)
wind_daily = _daily(wind_df, 'spp_wind_total_forecast_f', 'wind_forecast')

load_df  = load.Load(OPEX).get_total_load(start_dt, end_dt, var_spec=['f'], impute=True)
load_daily = _daily(load_df, 'spp_load_total_forecast_f', 'load_forecast')

go_df    = genoutage.GenOutage(OPEX).get_genoutage_by_level(start_dt, end_dt, var_spec=['f'], area_list=['SPP'])
go_col   = [c for c in go_df.columns if c.endswith('_forecast_f')][0]
genoutage_daily = _daily(go_df, go_col, 'genoutage_forecast')

# --- ice price forecast (INDIANAHUB proxy for SPP, node 636) ---
ice_df, _ = ice_elec_price_vh.get_data_and_mapping_for_ice_elec(
    [636], OPEX, ['INDIANAHUB'], start_dt, end_dt, var_spec=['f'], impute=True)
ice_daily = _daily(ice_df, 'INDIANAHUB_ice_elec_price_forecast_f', 'ice_price_forecast')

# --- combine all daily physical variables ---
phys_daily = (wind_daily
              .merge(load_daily,      on='dt', how='outer')
              .merge(genoutage_daily, on='dt', how='outer')
              .merge(ice_daily,       on='dt', how='outer'))
print('phys_daily:', phys_daily.shape)
display(phys_daily.head())

# --- merge onto train / holdout (dt-level) ---
train_df['dt']   = train_df['dt'].astype(str)
holdout_df['dt'] = holdout_df['dt'].astype(str)
train_df   = train_df.merge(phys_daily, on='dt', how='left')
holdout_df = holdout_df.merge(phys_daily, on='dt', how='left')
print('train_df  :', train_df.shape, '| holdout_df:', holdout_df.shape)
train_df.head()

phys date range: 2021-06-02 -> 2026-05-19
phys_daily: (1813, 5)


,dt,wind_forecast,load_forecast,genoutage_forecast,ice_price_forecast
0,2021-06-02,1916.375000,26712.708333,11762.216667,26.790000
1,2021-06-03,3768.707917,28603.416667,11286.708333,28.130000
2,2021-06-04,9523.051667,30291.458333,11359.608333,29.570000
3,2021-06-05,13756.800417,29792.250000,11398.025000,29.783333
4,2021-06-06,13693.260000,29196.708333,11431.691667,34.360000


train_df  : (582105, 82) | holdout_df: (222843, 82)


,constraint_family_num,dt,long_profit,short_profit,long_profit_sum_3d,short_profit_sum_3d,long_profit_sum_7d,short_profit_sum_7d,long_profit_sum_1m,short_profit_sum_1m,...,rt_spike_1m,rt_spike_3m,rt_spike_1y,rt_spike_3y,KV,KV_group,wind_forecast,load_forecast,genoutage_forecast,ice_price_forecast
0,7,2021-07-01,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,-852.205775,161.0,04_345,3450.68,34594.75,8961.054167,35.99
1,11,2021-07-01,0.0,0.0,0.000000,0.000000,0.000000,0.000000,28232.886036,-77974.874419,...,-2829.423183,-2894.467394,-1825.074444,-1841.973281,161.0,04_345,3450.68,34594.75,8961.054167,35.99
2,27,2021-07-01,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,-909.199441,NaN,03_138,3450.68,34594.75,8961.054167,35.99
3,30,2021-07-01,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,-747.833083,138.0,03_138,3450.68,34594.75,8961.054167,35.99
4,41,2021-07-01,0.0,0.0,9333.357069,-1297.486966,11249.508162,-2176.316475,965094.429461,-593286.123159,...,-808.141129,-1218.259754,-2271.628633,-2520.698961,115.0,02_<=115,3450.68,34594.75,8961.054167,35.99


## Driver analysis — what moves long & short profit (family-day level)

Goal: rank the variables that drive **short_profit** (loss side) and **long_profit**, both as a
*level* (regression) and as *tail-loss events* (`short_profit < THRESH`), then translate the top
stable drivers into cut rules. Importance is judged on the **holdout** split (out-of-sample) and
cross-checked by SHAP **and** permutation importance — only drivers that agree are trusted.

**Leakage guard:** same-day realized `dam`/`rtm` are excluded (they are contemporaneous with the
loss). Only lagged/rolling history and pre-bid forecasts (FlowRatio, wind/load/genoutage/ice) are used.

In [6]:
# === 1. feature setup (family-day level) ===
KEYS    = ['dt', 'constraint_family_num']
TARGETS = ['short_profit', 'long_profit']
# same-day realized / id columns -> EXCLUDE (leakage or non-features)
LEAK = ['dam', 'rtm', 'da_mvalue', 'rt_mvalue', 'bad_short', 'rt_da', 'rt_binds',
        'oops_constraint_num', 'monitored_clean', 'contingency_clean',
        'monitored_name', 'node_name']

num_cols = train_df.select_dtypes('number').columns.tolist()
features = [c for c in num_cols if c not in KEYS + TARGETS + LEAK]
CATS     = [c for c in ['KV_group'] if c in train_df.columns]   # categorical driver(s)

# tag features so you can separate "it was bad recently" from real conditions
persistence = [c for c in features if 'profit_sum' in c]
condition   = [c for c in features if c not in persistence]

print(f"{len(features)} numeric features (+{len(CATS)} categorical: {CATS})")
print(f"\nPERSISTENCE ({len(persistence)}): {persistence}")
print(f"\nCONDITION   ({len(condition)}): {condition}")

# tail-loss flag (same threshold used in the source notebook)
THRESH = -50000
for d in (train_df, holdout_df):
    d['bad_short'] = (d['short_profit'] < THRESH).astype(int)
print(f"\nbad_short rate  train={train_df['bad_short'].mean():.4f}  holdout={holdout_df['bad_short'].mean():.4f}")

72 numeric features (+1 categorical: ['KV_group'])

PERSISTENCE (10): ['long_profit_sum_3d', 'short_profit_sum_3d', 'long_profit_sum_7d', 'short_profit_sum_7d', 'long_profit_sum_1m', 'short_profit_sum_1m', 'long_profit_sum_3m', 'short_profit_sum_3m', 'long_profit_sum_1y', 'short_profit_sum_1y']

CONDITION   (62): ['FlowRatio', 'ShadowPrice_min', 'ShadowPrice_sum', 'MinFlowLimit', 'MaxFlowLimit', 'dam_lag1', 'rtm_lag1', 'dam_sum_7d', 'rtm_sum_7d', 'dam_sum_1m', 'rtm_sum_1m', 'dam_sum_3m', 'rtm_sum_3m', 'dam_sum_1y', 'rtm_sum_1y', 'dam_sum_3y', 'rtm_sum_3y', 'rtda_1d', 'rtda_sum_7d', 'rtda_avg_7d', 'rtda_max_7d', 'rtda_min_7d', 'rtda_sum_1m', 'rtda_avg_1m', 'rtda_max_1m', 'rtda_min_1m', 'rtda_sum_3m', 'rtda_avg_3m', 'rtda_max_3m', 'rtda_min_3m', 'rtda_sum_1y', 'rtda_avg_1y', 'rtda_max_1y', 'rtda_min_1y', 'rtda_sum_3y', 'rtda_avg_3y', 'rtda_max_3y', 'rtda_min_3y', 'rtda_std_1m', 'rtda_std_3m', 'rtda_std_1y', 'rtda_std_3y', 'rt_max_7d', 'rt_max_1m', 'rt_max_3m', 'rt_max_1y', 'rt_max_3y', '

In [7]:
# === 2. model + importance helper (SHAP + permutation, judged on holdout) ===
import numpy as np, pandas as pd, lightgbm as lgb, shap
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, roc_auc_score

SHAP_N = 50_000   # sample holdout for SHAP/permutation speed

def _xy(df, target):
    X = df[features + CATS].copy()
    for c in CATS: X[c] = X[c].astype('category')
    y = df[target]
    m = y.notna()
    return X[m], y[m]

def driver_analysis(target, task='reg'):
    Xtr, ytr = _xy(train_df, target)
    Xho, yho = _xy(holdout_df, target)
    if task == 'reg':
        model = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.03, num_leaves=31,
                                  subsample=0.8, colsample_bytree=0.8, min_child_samples=50,
                                  random_state=0, n_jobs=-1, verbose=-1)
        model.fit(Xtr, ytr, categorical_feature=CATS)
        print(f"[{target}] R2  train={r2_score(ytr, model.predict(Xtr)):.3f}  "
              f"holdout={r2_score(yho, model.predict(Xho)):.3f}")
        scoring = 'r2'
    else:
        model = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.03, num_leaves=31,
                                   subsample=0.8, colsample_bytree=0.8, min_child_samples=50,
                                   random_state=0, n_jobs=-1, verbose=-1, class_weight='balanced')
        model.fit(Xtr, ytr, categorical_feature=CATS)
        print(f"[{target}] AUC train={roc_auc_score(ytr, model.predict_proba(Xtr)[:,1]):.3f}  "
              f"holdout={roc_auc_score(yho, model.predict_proba(Xho)[:,1]):.3f}")
        scoring = 'roc_auc'

    Xs = Xho.sample(min(SHAP_N, len(Xho)), random_state=0)
    ys = yho.loc[Xs.index]

    sv = shap.TreeExplainer(model).shap_values(Xs)
    if isinstance(sv, list): sv = sv[-1]
    elif getattr(sv, 'ndim', 2) == 3: sv = sv[:, :, -1]

    perm = permutation_importance(model, Xs, ys, scoring=scoring,
                                  n_repeats=3, random_state=0, n_jobs=-1)
    imp = pd.DataFrame({'shap': np.abs(sv).mean(0),
                        'perm': perm.importances_mean}, index=Xs.columns)
    imp['shap_rank'] = imp['shap'].rank(ascending=False)
    imp['perm_rank'] = imp['perm'].rank(ascending=False)
    imp['mean_rank'] = imp[['shap_rank', 'perm_rank']].mean(1)
    imp = imp.sort_values('mean_rank')
    return {'model': model, 'shap': sv, 'X': Xs, 'imp': imp}

short_res = driver_analysis('short_profit', 'reg')
long_res  = driver_analysis('long_profit',  'reg')
bad_res   = driver_analysis('bad_short',     'clf')

print('\n=== top drivers of SHORT profit (level) ==='); display(short_res['imp'].head(15).round(3))
print('=== top drivers of LONG profit (level) ===');   display(long_res['imp'].head(15).round(3))
print('=== top drivers of BAD-SHORT tail events ==='); display(bad_res['imp'].head(15).round(3))

[short_profit] R2  train=0.366  holdout=-0.021
[long_profit] R2  train=0.349  holdout=0.035
[bad_short] AUC train=0.982  holdout=0.938


/opt/venvs/prod-py312/lib/python3.12/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(



=== top drivers of SHORT profit (level) ===


,shap,perm,shap_rank,perm_rank,mean_rank
long_profit_sum_1y,506.035,0.031,1.0,2.0,1.5
short_profit_sum_1m,471.961,0.012,3.0,4.0,3.5
short_profit_sum_3d,307.610,0.031,8.0,1.0,4.5
dam_sum_7d,480.835,0.003,2.0,13.0,7.5
rtda_sum_1y,229.161,0.006,14.0,6.0,10.0
long_profit_sum_3m,347.226,0.003,6.0,15.0,10.5
rtm_lag1,243.806,0.003,11.0,12.0,11.5
rtda_sum_3y,198.876,0.004,15.0,9.0,12.0
load_forecast,151.515,0.023,21.0,3.0,12.0
rt_nz_cnt_1m,176.871,0.005,19.0,7.0,13.0


=== top drivers of LONG profit (level) ===


,shap,perm,shap_rank,perm_rank,mean_rank
long_profit_sum_1y,666.771,0.009,1.0,1.0,1.0
long_profit_sum_3m,402.465,0.006,3.0,3.0,3.0
dam_lag1,340.963,0.004,6.0,4.0,5.0
rt_spike_1m,311.891,0.003,8.0,8.0,8.0
rtda_sum_3y,154.534,0.007,20.0,2.0,11.0
FlowRatio,329.858,0.001,7.0,16.0,11.5
dam_sum_7d,165.062,0.003,17.0,7.0,12.0
rtda_sum_7d,275.633,0.002,11.0,14.0,12.5
rtda_sum_3m,204.162,0.002,15.0,12.0,13.5
short_profit_sum_7d,152.908,0.004,22.0,6.0,14.0


=== top drivers of BAD-SHORT tail events ===


,shap,perm,shap_rank,perm_rank,mean_rank
FlowRatio,0.601,0.021,1.0,1.0,1.0
rtm_sum_1m,0.296,0.005,3.0,3.0,3.0
rtda_std_1m,0.388,0.005,2.0,4.0,3.0
wind_forecast,0.289,0.007,4.0,2.0,3.0
rtm_sum_7d,0.144,0.003,7.0,5.0,6.0
rtda_min_3m,0.136,0.001,8.0,12.0,10.0
rt_nz_cnt_1y,0.145,0.001,6.0,14.0,10.0
rtm_sum_1y,0.093,0.001,10.0,11.0,10.5
rtda_std_3y,0.081,0.001,12.0,10.0,11.0
rt_spike_3y,0.148,0.001,5.0,22.0,13.5


In [ ]:
# === 3. SHAP summary plots (direction + magnitude) ===
import matplotlib.pyplot as plt
for name, res in [('short_profit', short_res), ('long_profit', long_res), ('bad_short', bad_res)]:
    shap.summary_plot(res['shap'], res['X'], max_display=15, show=False)
    plt.title(f'SHAP — {name}'); plt.tight_layout(); plt.show()

In [ ]:
# === 4. control tradeoff: cutting the worst tail of a driver ===
# For a candidate driver, drop the family-days in its 'bad' tail and measure
# short loss avoided vs long profit sacrificed. Set high_is_bad from the SHAP plot.
def cut_tradeoff(df, feat, q=0.10, high_is_bad=True):
    s = df[feat]
    thr = s.quantile(1 - q) if high_is_bad else s.quantile(q)
    cut = (s >= thr) if high_is_bad else (s <= thr)
    return pd.Series({
        'feature': feat, 'threshold': round(thr, 3), 'cut_frac': round(cut.mean(), 3),
        'short_loss_avoided': -df.loc[cut, 'short_profit'].sum(),   # >0 = loss removed
        'long_profit_lost':    df.loc[cut, 'long_profit'].sum(),    # >0 = profit forgone
        'net_benefit':       -df.loc[cut, 'short_profit'].sum() - df.loc[cut, 'long_profit'].sum(),
    })

# evaluate the top tail-event drivers on the holdout (set high_is_bad per SHAP direction)
top_drivers = bad_res['imp'].head(6).index.tolist()
tradeoff = pd.DataFrame([cut_tradeoff(holdout_df, f, q=0.10, high_is_bad=True)
                         for f in top_drivers if f in holdout_df.columns])
display(tradeoff.sort_values('net_benefit', ascending=False))

In [8]:
# === where does the BAD-SHORT forest split? ===
import numpy as np, pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text

bad_model = bad_res['model']
top_feats = bad_res['imp'].head(8).index.tolist()

# ---- A. exact split thresholds from the trained forest (gain-weighted) ----
tdf    = bad_model.booster_.trees_to_dataframe()
splits = tdf[tdf['split_feature'].notna()].copy()
splits['thr_num'] = pd.to_numeric(splits['threshold'], errors='coerce')  # NaN = categorical split

print("Split thresholds per top driver (gain-weighted, from the actual forest):\n")
rows = []
for f in top_feats:
    s = splits[splits['split_feature'] == f]
    if s.empty:
        rows.append({'feature': f, 'note': 'not used as a split'}); continue
    sn = s.dropna(subset=['thr_num'])
    if sn.empty:                                   # categorical (e.g. KV_group)
        rows.append({'feature': f, 'n_splits': len(s), 'note': 'categorical split'}); continue
    w = sn['split_gain']
    gain_wtd_thr = np.average(sn['thr_num'], weights=w)        # typical split point
    band = sn.groupby(pd.cut(sn['thr_num'], 10))['split_gain'].sum().idxmax()  # highest-gain band
    rows.append({'feature': f, 'n_splits': len(sn),
                 'total_gain': round(w.sum(), 1),
                 'median_thr': round(sn['thr_num'].median(), 4),
                 'gain_wtd_thr': round(gain_wtd_thr, 4),
                 'top_gain_band': str(band)})
display(pd.DataFrame(rows))

# top single split points overall (which feature@threshold buys the most gain)
print("\nTop 15 individual splits by gain:")
display(splits.dropna(subset=['thr_num'])
              .sort_values('split_gain', ascending=False)
              [['split_feature', 'thr_num', 'split_gain', 'count']].head(15).round(4))

# ---- B. shallow surrogate tree -> readable if/then rules for the forest's risk ----
Xm = holdout_df[features + CATS].copy()
for c in CATS: Xm[c] = Xm[c].astype('category')
risk = bad_model.predict_proba(Xm)[:, 1]                 # forest risk score
hot  = (risk > np.quantile(risk, 0.90)).astype(int)      # explain the top-decile-risk region

Xt = holdout_df[features + CATS].copy()
for c in CATS: Xt[c] = Xt[c].astype('category').cat.codes
surr = DecisionTreeClassifier(max_depth=3, min_samples_leaf=500, random_state=0).fit(Xt, hot)
print(f"\nSurrogate tree (depth 3) — explains the forest's top-decile risk | "
      f"fidelity acc={surr.score(Xt, hot):.3f}\n")
print(export_text(surr, feature_names=list(Xt.columns), max_depth=3))

Split thresholds per top driver (gain-weighted, from the actual forest):



,feature,n_splits,total_gain,median_thr,gain_wtd_thr,top_gain_band
0,FlowRatio,929,703836.5,0.7122,0.7162,"(0.675, 0.779]"
1,rtm_sum_1m,235,3453958.6,-1560.8736,-245.9394,"(-12109.197, -1.0000000000000001e-35]"
2,rtda_std_1m,246,1197566.7,47.0202,47.5194,"(-1.869, 186.882]"
3,wind_forecast,799,211632.1,12935.9373,12483.5781,"(8036.136, 10668.14]"
4,rtm_sum_7d,168,501454.5,-499.3133,-370.2992,"(-3159.47, -1.0000000000000001e-35]"
5,rtda_min_3m,160,45341.4,-958.8832,-1585.7796,"(-1955.545, -1.0000000000000001e-35]"
6,rt_nz_cnt_1y,376,106217.8,31.5000,23.9223,"(1.222, 29.3]"
7,rtm_sum_1y,281,75223.9,-10480.9818,-20894.7025,"(-58216.228, -1.0000000000000001e-35]"



Top 15 individual splits by gain:


,split_feature,thr_num,split_gain,count
0,rtm_sum_1m,-122.1639,269978.0,582105
61,rtm_sum_1m,-122.1639,254173.0,582105
122,rtm_sum_1m,-122.1639,239559.0,582105
183,rtm_sum_1m,-122.1639,226014.0,582105
244,rtm_sum_1m,-71.9241,213443.0,582105
305,rtm_sum_1m,-122.1639,201733.0,582105
366,rtm_sum_1m,-71.9241,190865.0,582105
427,rtm_sum_1m,-71.9241,180661.0,582105
488,rt_max_1m,-103.3107,170717.0,582105
549,rtm_sum_1m,-71.9241,162154.0,582105



Surrogate tree (depth 3) — explains the forest's top-decile risk | fidelity acc=0.948

|--- rtm_sum_7d <= -285.29
|   |--- FlowRatio <= 0.79
|   |   |--- short_profit_sum_3d <= -18369.42
|   |   |   |--- class: 1
|   |   |--- short_profit_sum_3d >  -18369.42
|   |   |   |--- class: 0
|   |--- FlowRatio >  0.79
|   |   |--- wind_forecast <= 5706.84
|   |   |   |--- class: 1
|   |   |--- wind_forecast >  5706.84
|   |   |   |--- class: 1
|--- rtm_sum_7d >  -285.29
|   |--- rtm_sum_1m <= -236.92
|   |   |--- FlowRatio <= 0.83
|   |   |   |--- class: 0
|   |   |--- FlowRatio >  0.83
|   |   |   |--- class: 1
|   |--- rtm_sum_1m >  -236.92
|   |   |--- ShadowPrice_sum <= -1119.85
|   |   |   |--- class: 0
|   |   |--- ShadowPrice_sum >  -1119.85
|   |   |   |--- class: 0

